# 🏥 Clinical Video Analysis — Treinamento YOLOv8
Treina 4 modelos especializados por contexto clínico:
- **Cirurgia**: detecção de sangramento
- **Consulta**: sinais de dor/desconforto
- **Fisioterapia**: análise de postura e movimento
- **Triagem**: linguagem corporal indicativa de violência

## 1. Instalar Dependências

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'kagglehub', 'ultralytics', 'pyyaml'], check=True)
print('✅ Dependências instaladas')

✅ Dependências instaladas


## 2. Verificar GPU

In [28]:
import torch
DEVICE = 0
if torch.cuda.is_available():
    print(f"GPU disponível: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = 0
else:
    print("GPU não encontrada, usando CPU (treinamento mais lento)")
    DEVICE = 'cpu'


GPU disponível: NVIDIA GeForce RTX 3070
VRAM: 8.6 GB


## 3. Baixar Datasets

In [11]:
import kagglehub, os, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from roboflow import Roboflow

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

paths = {}

print("📥 DOWNLOAD DATASETS FEMHEALTH")
print("=" * 55)

# 1. LAPAROSCOPIA ROBFLOW → data/cirurgia/
print("🔄 1/4 CIRURGIA: Laparoscopia...")
rf = Roboflow(api_key="qYZBO5fqNIjOlDK6jH4o")  # SUA chave!
project = rf.workspace("laparoscopic-yolo").project("laparoscopy")
dataset_obj = project.version(14).download("yolov8")  # Retorna Dataset obj

# Extrai path real (fix TypeError)
temp_path = dataset_obj.location  # ← FIX: .location da Dataset obj
cirurgia_path = os.path.join(DATA_DIR, "cirurgia")
shutil.move(temp_path, cirurgia_path)
paths['cirurgia'] = cirurgia_path
print("✅ CIRURGIA:  data/cirurgia/ ✓")

# 2. KAGGLE → data/{context}/
print("\n🔄 2-4/4 KAGGLE...")
DATASETS_KAGGLE = {
    'consulta':     'coder98/emotionpain',
    'fisioterapia': 'sulaimanmuhammed/wlu-rehabilitation-posture',
    'triagem':      'simuletic/cctv-aggressive-poses-and-fight-detection-dataset',
}

def download_to_data(context, dataset_id):
    try:
        temp_path = kagglehub.dataset_download(dataset_id)
        target_path = os.path.join(DATA_DIR, context)
        shutil.move(temp_path, target_path)
        return context, target_path, None
    except Exception as e:
        return context, None, str(e)

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(download_to_data, ctx, ds): ctx for ctx, ds in DATASETS_KAGGLE.items()}
    for future in as_completed(futures):
        context, path, error = future.result()
        if error:
            print(f"❌ {context:>11}: {error}")
        else:
            paths[context] = path
            print(f"✅ {context:>11}: OK")

print("\n✅ ESTRUTURA:")
print(f"data/")
for ctx in ['cirurgia', 'consulta', 'fisioterapia', 'triagem']:
    status = "✅" if ctx in paths else "❌"
    print(f"├── {ctx}/  {status}")


📥 DOWNLOAD DATASETS FEMHEALTH
🔄 1/4 CIRURGIA: Laparoscopia...
loading Roboflow workspace...
loading Roboflow project...
✅ CIRURGIA:  data/cirurgia/ ✓

🔄 2-4/4 KAGGLE...
✅    consulta: OK
✅     triagem: OK
✅ fisioterapia: OK

✅ ESTRUTURA:
data/
├── cirurgia/  ✅
├── consulta/  ✅
├── fisioterapia/  ✅
├── triagem/  ✅


## 4. Inspecionar Estrutura dos Datasets

In [22]:
import os

def inspect_dataset(path, name):
    """Status claro COM emojis explicados"""
    if not os.path.exists(path):
        return f"{name:12} ❌ Pasta inexistente"
    
    imgs = sum(1 for root, _, files in os.walk(path) 
               for f in files if f.lower().endswith(('.jpg','.jpeg','.png')))
    
    labels = sum(1 for root, _, files in os.walk(path) 
                 for f in files if f.endswith('.txt'))
    
    videos = sum(1 for root, _, files in os.walk(path) 
                 for f in files if f.lower().endswith(('.mp4','.avi')))
    
    yaml_ok = os.path.exists(os.path.join(path, 'data.yaml'))
    
    # STATUS COM EXPLICAÇÃO
    if yaml_ok and imgs > 0 and labels > 0 and abs(imgs-labels) < 20:
        status = "✅ YOLOv8 pronto (tem yaml+imgs+labels)"
    elif videos > 0:
        status = "🎥 Vídeos crus (precisa extrair frames)" 
    elif imgs > 0 and labels == 0:
        status = "🖼️ Apenas imagens (classification ou rotular)"
    elif imgs == 0 and labels == 0:
        status = "📂 Vazio (precisa download/preprocess)"
    else:
        status = "⚠️ Inconsistente (imgs≠labels)"
    
    return f"{name:12} | {imgs:>4}i {labels:>3}l {videos:>3}v | {status}"

print("📊 DATASETS FEMHEALTH")
print("="*70)
print("i=imagens l=labels v=vídeos")
print("-"*70)

for ctx, path in paths.items():
    print(inspect_dataset(path, ctx.upper()))


📊 DATASETS FEMHEALTH
i=imagens l=labels v=vídeos
----------------------------------------------------------------------
CIRURGIA     | 1169i 1171l   0v | ✅ YOLOv8 pronto (tem yaml+imgs+labels)
CONSULTA     | 48398i 96796l   0v | ⚠️ Inconsistente (imgs≠labels)
TRIAGEM      |  103i 103l   0v | ⚠️ Inconsistente (imgs≠labels)
FISIOTERAPIA |    0i   0l 257v | 🎥 Vídeos crus (precisa extrair frames)


In [20]:
import os, yaml
from pathlib import Path

def analyze_dataset(path):
    """Análise YOLOv8 completa e precisa"""
    p = Path(path)
    if not p.exists():
        return {'error': 'Pasta não existe'}
    
    stats = {
        'videos': sum(1 for x in p.rglob('*') if x.suffix.lower() in {'.mp4','.mov','.avi'}),
        'images': sum(1 for x in p.rglob('*') if x.suffix.lower() in {'.jpg','.jpeg','.png'}),
        'labels_txt': sum(1 for x in p.rglob('*.txt')),
        'yaml': False,
        'train_split': bool(p / 'train'),
        'valid_split': bool(p / 'val'),
        'classes_yaml': [],
        'train_path_ok': False,
        'val_path_ok': False
    }
    
    # YAML análise detalhada
    yaml_file = next(p.rglob('data.yaml'), None)
    if yaml_file and yaml_file.exists():
        stats['yaml'] = True
        try:
            with open(yaml_file) as f:
                data = yaml.safe_load(f)
                stats['classes_yaml'] = data.get('names', [])
                train_rel = data.get('train', '')
                val_rel = data.get('val', '')
                stats['train_path_ok'] = (p / train_rel).exists()
                stats['val_path_ok'] = (p / val_rel).exists()
        except:
            stats['yaml'] = False
    
    stats['imgs_labels_ratio'] = 'OK' if abs(stats['images'] - stats['labels_txt']) < 20 else f"{stats['images']}/{stats['labels_txt']}"
    return stats

# ════════════════════════════════════════════════════════════════
print("🔍 VERIFICAÇÃO YOLOv8 DETALHADA")
print("="*75)
print("imgs | lbls | yaml | train/val | classes | status")
print("-"*75)

for context, path in paths.items():
    stats = analyze_dataset(path)
    
    if 'error' in stats:
        print(f"{context.upper():<12} | ❌ {stats['error']}")
        continue
    
    # Linha única clara
    yaml_status = "✅" if stats['yaml'] else "❌"
    train_status = "✅" if stats['train_split'] and stats['train_path_ok'] else "❌"
    val_status = "✅" if stats['valid_split'] and stats['val_path_ok'] else "❌"
    
    ready = stats['yaml'] and stats['images'] > 0 and stats['labels_txt'] > 0 and stats['train_path_ok'] and stats['val_path_ok']
    status = "🚀 PRONTO TREINO" if ready else "⚠️ PROBLEMAS"
    
    classes_preview = str(stats['classes_yaml'])[:30] + "..." if stats['classes_yaml'] else "N/A"
    
    print(f"{context.upper():<12} | {stats['images']:>4}/{stats['labels_txt']:>4} | {yaml_status} | {train_status}/{val_status} | {classes_preview:<30} | {status}")

print("\n" + "="*75)
print("🎯 COMANDOS TREINO PRONTOS:")
for ctx in ['cirurgia', 'triagem']:
    if ctx in paths:
        stats = analyze_dataset(paths[ctx])
        if stats.get('yaml') and stats['train_path_ok']:
            print(f"  YOLO('yolov8n.pt').train(data='{paths[ctx]}', epochs=50)")
        else:
            print(f"  {ctx.upper()}: ⚠️ PREPROCESS primeiro")


🔍 VERIFICAÇÃO YOLOv8 DETALHADA
imgs | lbls | yaml | train/val | classes | status
---------------------------------------------------------------------------
CIRURGIA     | 1169/1171 | ✅ | ❌/❌ | ['Allis', 'Bag', 'Cautery', 'F... | ⚠️ PROBLEMAS
CONSULTA     | 48398/96796 | ❌ | ❌/❌ | N/A                            | ⚠️ PROBLEMAS
TRIAGEM      |  103/ 103 | ✅ | ❌/❌ | {0: 'person'}...               | ⚠️ PROBLEMAS
FISIOTERAPIA |    0/   0 | ❌ | ❌/❌ | N/A                            | ⚠️ PROBLEMAS

🎯 COMANDOS TREINO PRONTOS:
  CIRURGIA: ⚠️ PREPROCESS primeiro
  TRIAGEM: ⚠️ PREPROCESS primeiro


## 5. Gerar data.yaml para Cada Contexto

In [23]:
import os, cv2, yaml, random, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

DATASET_OUT = Path("datasets/yolov8")
DATASET_OUT.mkdir(parents=True, exist_ok=True)

def safe_mkdir(p):
    """Cria diretório com parents=True seguro"""
    p.mkdir(parents=True, exist_ok=True)

def extract_frames(video_path, out_dir, step=5):
    cap = cv2.VideoCapture(str(video_path))
    frame_id = saved = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_id % step == 0:
            out = out_dir / f"{video_path.stem}_{frame_id:06d}.jpg"
            cv2.imwrite(str(out), frame)
            saved += 1
        frame_id += 1
    cap.release()
    return saved

def split_list(items, train_ratio=0.8):
    random.shuffle(items)
    n = int(len(items) * train_ratio)
    return items[:n], items[n:]

# =============================================================================
# PROCESSADORES (com error handling)
# =============================================================================
def process_cirurgia(path):
    print("🔧 CIRURGIA: Laparoscopia Roboflow")
    src = Path(path)
    out = DATASET_OUT / "cirurgia"
    safe_mkdir(out)
    shutil.copytree(src, out, dirs_exist_ok=True)
    
    yaml_path = out / "data.yaml"
    if yaml_path.exists():
        with open(yaml_path) as f:
            data = yaml.safe_load(f)
        print(f"✅ Classes: {data.get('names', 'N/A')}")
    return "cirurgia", str(yaml_path)

def process_triagem(path):
    print("🔧 TRIAGEM: Aggressive poses")
    src = Path(path) / "Aggressive_Poses_Dataset"
    out = DATASET_OUT / "triagem"
    safe_mkdir(out)
    
    # Copia raw
    raw_out = out / "raw"
    safe_mkdir(raw_out)
    shutil.copytree(src, raw_out, dirs_exist_ok=True)
    
    # YAML
    classes = sorted(set(int(float(l.split()[0])) for lbl in raw_out.rglob("*.txt") 
                        for l in open(lbl) if l.strip()))
    data_yaml = {
        "path": str(out),
        "train": "raw/images/train",
        "val": "raw/images/val",
        "nc": len(classes),
        "names": {i: f"person_pose_{i}" for i in classes}
    }
    yaml_path = out / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)
    return "triagem", str(yaml_path)

def process_fisioterapia(path):
    print("🔧 FISIOTERAPIA: Vídeos → frames")
    src = Path(path)
    out = DATASET_OUT / "fisioterapia"
    safe_mkdir(out)
    
    # Encontra vídeos
    videos = []
    for ext in ['*.mp4', '*.avi', '*.mov']:
        videos.extend(src.rglob(ext))
    print(f"🎥 {len(videos)} vídeos")
    
    if not videos:
        print("⚠️ Sem vídeos encontrados")
        return "fisioterapia", str(out)
    
    # Extrai frames
    temp_dir = out / "temp_frames"
    safe_mkdir(temp_dir)
    
    total_frames = 0
    with ThreadPoolExecutor(max_workers=2) as ex:  # Reduz workers
        futures = {ex.submit(extract_frames, v, temp_dir): v for v in videos[:20]}  # Limita 20 vídeos primeiro
        for f in futures:
            total_frames += f.result()
    
    print(f"🖼️ {total_frames} frames extraídos")
    
    # Organiza por classe (pastas)
    classes = [d for d in temp_dir.iterdir() if d.is_dir()]
    for cls in classes[:3]:  # Primeiras 3 classes
        frames = list(cls.glob("*.jpg"))
        if frames:
            train, val = split_list(frames)
            safe_mkdir(out / "train" / cls.name)
            safe_mkdir(out / "val" / cls.name)
            for f in train[:50]:  # Limita por classe
                shutil.move(str(f), out / "train" / cls.name)
            for f in val[:10]:
                shutil.move(str(f), out / "val" / cls.name)
    
    shutil.rmtree(temp_dir, ignore_errors=True)
    
    # YAML classification
    class_names = [c.name for c in out.iterdir() if c.is_dir() and (c/"train").exists()]
    data_yaml = {"path": str(out), "train": "train", "val": "val", "nc": len(class_names), "names": class_names}
    yaml_path = out / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)
    
    return "fisioterapia", str(yaml_path)

# EXECUTA
tasks = {
    "cirurgia": process_cirurgia,
    "triagem": process_triagem, 
    "fisioterapia": process_fisioterapia
}

yamls = {}
print("🚀 PREPROCESSAMENTO...")
for name, func in tasks.items():
    if name in paths:
        try:
            result = func(paths[name])
            yamls[result[0]] = result[1]
            print(f"✅ {name}")
        except Exception as e:
            print(f"❌ {name}: {e}")

print("\n📄 YAMLs:", yamls)


🚀 PREPROCESSAMENTO...
🔧 CIRURGIA: Laparoscopia Roboflow
✅ Classes: ['Allis', 'Bag', 'Cautery', 'Forceps', 'Gallbladder', 'Liver', 'Suction']
✅ cirurgia
🔧 TRIAGEM: Aggressive poses
✅ triagem
🔧 FISIOTERAPIA: Vídeos → frames
🎥 339 vídeos
🖼️ 578 frames extraídos
✅ fisioterapia

📄 YAMLs: {'cirurgia': 'datasets\\yolov8\\cirurgia\\data.yaml', 'triagem': 'datasets\\yolov8\\triagem\\data.yaml', 'fisioterapia': 'datasets\\yolov8\\fisioterapia\\data.yaml'}


In [27]:
import yaml
from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def resolve_path(root: Path, rel_path: str) -> Path:
    """Resolve path relativo do YAML com fallback inteligente"""
    if not rel_path:
        return None

    p = Path(rel_path)

    if p.is_absolute():
        return p

    resolved = (root / p).resolve()

    if resolved.exists():
        return resolved

    # fallback comum em datasets roboflow
    alt = (root.parent / p).resolve()
    if alt.exists():
        return alt

    return resolved


def count_images(path: Path):
    return sum(1 for f in path.rglob("*") if f.suffix.lower() in IMAGE_EXTS)


def count_labels(path: Path):
    return sum(1 for f in path.rglob("*.txt"))


def normalize_classes(names):
    """Normaliza classes (list ou dict)"""
    if isinstance(names, dict):
        return list(names.values())
    if isinstance(names, list):
        return names
    return []


def analyze_yolo_output(yaml_path):

    yaml_path = Path(yaml_path)

    if not yaml_path.exists():
        return {"error": "data.yaml não encontrado"}

    try:
        with open(yaml_path) as f:
            data = yaml.safe_load(f)
    except Exception:
        return {"error": "erro ao ler yaml"}

    root = yaml_path.parent

    train_dir = resolve_path(root, data.get("train"))
    val_dir = resolve_path(root, data.get("val"))

    stats = {
        "images_train": 0,
        "images_val": 0,
        "labels_train": 0,
        "labels_val": 0,
        "classes": normalize_classes(data.get("names")),
        "train_ok": False,
        "val_ok": False,
        "yaml_path": str(yaml_path),
    }

    if train_dir and train_dir.exists():
        stats["train_ok"] = True
        stats["images_train"] = count_images(train_dir)
        stats["labels_train"] = count_labels(train_dir)

    if val_dir and val_dir.exists():
        stats["val_ok"] = True
        stats["images_val"] = count_images(val_dir)
        stats["labels_val"] = count_labels(val_dir)

    stats["ready"] = (
        stats["train_ok"]
        and stats["val_ok"]
        and stats["images_train"] > 0
    )

    return stats


print("\n🔍 VERIFICAÇÃO DO PREPROCESSAMENTO")
print("=" * 90)
print("dataset       train(img/lbl)   val(img/lbl)   classes   status")
print("-" * 90)

results = {}

for name, yaml_path in yamls.items():

    stats = analyze_yolo_output(yaml_path)
    results[name] = stats

    if "error" in stats:
        print(f"{name:<12} ❌ {stats['error']}")
        continue

    train_info = f"{stats['images_train']}/{stats['labels_train']}"
    val_info = f"{stats['images_val']}/{stats['labels_val']}"

    classes_count = len(stats["classes"])

    status = "🚀 PRONTO" if stats["ready"] else "⚠️ PROBLEMA"

    print(
        f"{name:<12} "
        f"{train_info:>12}   "
        f"{val_info:>12}   "
        f"{classes_count:>7}   "
        f"{status}"
    )

print("\n🎯 COMANDOS DE TREINO:\n")

for name, stats in results.items():

    if stats.get("ready"):
        yaml_path = stats["yaml_path"]

        print(
            f"# {name}\n"
            f"from ultralytics import YOLO\n"
            f"model = YOLO('yolov8n.pt')\n"
            f"model.train(data='{yaml_path}', epochs=50)\n"
        )

    else:
        print(f"# {name} ⚠️ dataset incompleto\n")


🔍 VERIFICAÇÃO DO PREPROCESSAMENTO
dataset       train(img/lbl)   val(img/lbl)   classes   status
------------------------------------------------------------------------------------------
cirurgia              0/0            0/0         7   ⚠️ PROBLEMA
triagem               0/0            0/0         1   ⚠️ PROBLEMA
fisioterapia          0/0            0/0         0   ⚠️ PROBLEMA

🎯 COMANDOS DE TREINO:

# cirurgia ⚠️ dataset incompleto

# triagem ⚠️ dataset incompleto

# fisioterapia ⚠️ dataset incompleto



## 6. Treinar os Modelos

In [25]:
# ================================
# PIPELINE TREINO YOLOv8 - LAPAROSCOPIA FEMHEALTH (FIX CLASSIFY + SYNTAX)
# ================================

from ultralytics import YOLO
import os, glob, traceback
from pathlib import Path

# CONFIG
DEVICE = 0  # GPU ('cpu' se erro)
BASE_DIR = Path("models/yolov8_femhealth")
BASE_DIR.mkdir(parents=True, exist_ok=True)

resultados = {}
status = {}

def safe_train(name, base_model, data_path, task="detect", epochs=30):
    """Treina YOLOv8 com FIX classify + skip"""
    
    exp_dir = BASE_DIR / task / name
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    best_pt = exp_dir / "weights" / "best.pt"
    if best_pt.exists():
        print(f"✅ {name.upper()} SKIP - {best_pt}")
        resultados[name] = str(exp_dir)
        return "SKIP"
    
    # 🔥 FIX CLASSIFICATION: pasta raiz (não YAML)
    if task == "classify":
        data_folder = Path(data_path).parent
        data_path = str(data_folder)
        print(f"🔧 CLASSIFY pasta: {data_folder}")
    
    print(f"\n🚀 TREINO: {name.upper()} ({task})")
    print(f"📁 data: {data_path}")
    
    try:
        params = {
            "data": data_path,
            "epochs": epochs,
            "imgsz": 640 if task != "classify" else 224,
            "batch": 16 if task == "detect" else 32,
            "device": DEVICE,
            "project": str(BASE_DIR),
            "name": f"{task}/{name}",
            "exist_ok": True,
            "patience": 8,
            "workers": 2,
            "save_period": 10,
            "task": task,
            "augment": True,
            "val": True
        }
        
        model = YOLO(base_model)
        results = model.train(**params)
        
        resultados[name] = str(results.save_dir)
        print(f"✅ {name} → {results.save_dir}")
        return "OK"
        
    except Exception as e:
        print(f"❌ {name}: {e}")
        traceback.print_exc()
        return "ERROR"

# ================================
# EXECUÇÃO ORDENADA
# ================================
print("🔄 PIPELINE FEMHEALTH YOLOv8")

# 1️⃣ CIRURGIA LAPAROSCOPIA
if "cirurgia" in yamls:
    status["cirurgia"] = safe_train("laparoscopia", "yolov8n.pt", yamls["cirurgia"], "detect", 50)

# 2️⃣ TRIAGEM POSES
if "triagem" in yamls:
    status["triagem"] = safe_train("aggressive_poses", "yolov8n.pt", yamls["triagem"], "detect", 30)

# 3️⃣ FISIOTERAPIA POSTURAS
if "fisioterapia" in yamls:
    status["fisioterapia"] = safe_train("rehab_postures", "yolov8n-cls.pt", yamls["fisioterapia"], "classify", 30)

print("\n" + "="*70)
print("🎯 RESUMO FINAL")
print("="*70)
for name, st in status.items():
    path = resultados.get(name, "-")
    print(f"{name:<20} | {st:<10} | {path}")

print(f"\n🚀 INFERÊNCIA:")
print("cirurgia:  YOLO('models/yolov8_femhealth/detect/laparoscopia/weights/best.pt')")
print("triagem:   YOLO('models/yolov8_femhealth/detect/aggressive_poses/weights/best.pt')")
print("fisio:     YOLO('models/yolov8_femhealth/classify/rehab_postures/weights/best.pt').predict(img, task='classify')")


🔄 PIPELINE FEMHEALTH YOLOv8

🚀 TREINO: LAPAROSCOPIA (detect)
📁 data: datasets\yolov8\cirurgia\data.yaml
Ultralytics 8.4.19  Python-3.12.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets\yolov8\cirurgia\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

KeyboardInterrupt: 

## 7. Avaliar Métricas dos Modelos

In [ ]:
from ultralytics import YOLO

print(f'{'Contexto':<15} {'mAP50':>8} {'mAP50-95':>10} {'Precisão':>10} {'Recall':>8}')
print('-' * 55)

for context, model_path in resultados.items():
    model = YOLO(model_path)
    metrics = model.val(data=yamls[context], verbose=False)
    print(
        f'{context:<15}'
        f'{metrics.box.map50:>8.3f}'
        f'{metrics.box.map:>10.3f}'
        f'{metrics.box.mp:>10.3f}'
        f'{metrics.box.mr:>8.3f}'
    )


## 8. Testar com Imagem Real

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

os.makedirs('data/samples', exist_ok=True)

for context, model_path in resultados.items():
    # Busca primeira imagem disponível do dataset
    test_img = next(
        (os.path.join(root, f)
         for root, _, files in os.walk(paths[context])
         for f in files if f.endswith(('.jpg', '.png', '.jpeg'))),
        None
    )

    if not test_img:
        print(f'⚠️  Nenhuma imagem encontrada para {context}')
        continue

    model = YOLO(model_path)
    results = model(test_img, conf=0.25)
    out_path = f'data/samples/resultado_{context}.jpg'
    results[0].save(out_path)

    print(f'\n🔍 {context.upper()}')
    print(f'   Imagem: {os.path.basename(test_img)}')
    print(f'   Detecções: {len(results[0].boxes)}')
    display(Image(filename=out_path, width=400))


## 9. Resumo Final

In [ ]:
import os

print('\n📦 MODELOS TREINADOS:')
print('-' * 45)
for context in ['cirurgia', 'consulta', 'fisioterapia', 'triagem']:
    path = f'models/yolov8_custom/{context}.pt'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✅ {context:<15} {path} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ {context:<15} não treinado')

print('\n🚀 Próximo passo: uvicorn app.main:app --reload')
